# 04 — Model Architecture
**Step 18:** Custom Multimodal Attention Graph Neural Network (CMAGNN) design.

Components:
- Optical branch (Sentinel-2 spectral MLP)
- Radar branch (Sentinel-1 radar MLP)
- Feature fusion layer
- Graph Attention layer (GAT)
- Classification layer + Softmax

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Hyperparameters ───────────────────────────────────────────
N_OPTICAL  = 5    # B2, B3, B4, B8, B11
N_RADAR    = 3    # VV, VH, VV_VH_ratio
N_FEATURES = 8    # total
N_CLASSES  = 5
HIDDEN_DIM = 64
DROPOUT    = 0.3
GAT_HEADS  = 4

In [ ]:
# ── CMAGNN Model ─────────────────────────────────────────────

class ModalityBranch(nn.Module):
    """MLP branch to extract embeddings from one modality."""

    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class FusionLayer(nn.Module):
    """Attention-weighted fusion of two branch embeddings."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, 2),
            nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, opt_emb, rad_emb):
        combined = torch.cat([opt_emb, rad_emb], dim=-1)
        weights  = self.attn(combined)              # (N, 2)
        fused    = (weights[:, 0:1] * opt_emb +
                    weights[:, 1:2] * rad_emb)      # weighted sum
        return F.relu(self.proj(fused))


class CMAGNN(nn.Module):
    """
    Custom Multimodal Attention Graph Neural Network.

    Input  : node feature tensor x (N, 8) — first 5 = optical, last 3 = radar
    Output : class logits (N, n_classes)
    """

    def __init__(self, n_optical=5, n_radar=3, hidden_dim=64,
                 n_classes=5, dropout=0.3, gat_heads=4):
        super().__init__()

        # Modality branches
        self.optical_branch = ModalityBranch(n_optical, hidden_dim, dropout)
        self.radar_branch   = ModalityBranch(n_radar,   hidden_dim, dropout)

        # Fusion
        self.fusion = FusionLayer(hidden_dim)

        # Graph Attention layers
        self.gat1 = GATConv(hidden_dim, hidden_dim // gat_heads,
                            heads=gat_heads, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_dim, hidden_dim,
                            heads=1, dropout=dropout, concat=False)

        self.bn1     = nn.BatchNorm1d(hidden_dim)
        self.bn2     = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Classification head
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, edge_index):
        # Split modalities
        x_opt = x[:, :5]   # Sentinel-2 bands
        x_rad = x[:, 5:]   # Sentinel-1 bands + ratio

        # Branch embeddings
        opt_emb = self.optical_branch(x_opt)
        rad_emb = self.radar_branch(x_rad)

        # Fuse
        h = self.fusion(opt_emb, rad_emb)

        # GAT layer 1
        h = self.gat1(h, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.dropout(h)

        # GAT layer 2
        h = self.gat2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)

        # Classification
        return self.classifier(h)


# ── Instantiate & inspect ─────────────────────────────────────
model = CMAGNN(
    n_optical  = N_OPTICAL,
    n_radar    = N_RADAR,
    hidden_dim = HIDDEN_DIM,
    n_classes  = N_CLASSES,
    dropout    = DROPOUT,
    gat_heads  = GAT_HEADS
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Total trainable parameters: {total_params:,}')

# Save model class for import in training notebook
import torch
torch.save(model.state_dict(), '../outputs/model_init.pth')
print('✅ Initial weights saved to outputs/model_init.pth')